# Gold — Índice-proxy e indicadores de mercado

Lê a tabela `poc_b3_modernizacao.silver.cotacoes` (dados limpos, sem os registros em
quarentena) e calcula os indicadores de negócio do projeto.

**Camada gerada exclusivamente de forma automatizada** — sem edição manual em nenhuma
hipótese, especificamente para eliminar divergência de números entre diferentes
consumidores do dado (ver seção de governança em `docs/architecture.md`).

Indicadores calculados:
1. **Retorno diário por ticker**: `(preço atual - fechamento anterior) / fechamento anterior`
2. **Índice-proxy**: média simples do retorno diário dos 4 papéis
3. **Ranking de valorização**: ordena os papéis do maior para o menor retorno do dia
4. **Dispersão do dia**: desvio padrão dos 4 retornos diários

*Indicador futuro (pendente de pelo menos 2 dias de dados válidos): variação acumulada
do índice entre dias.*

**Entrada:** tabela `poc_b3_modernizacao.silver.cotacoes`
**Saída:** tabela `poc_b3_modernizacao.gold.indicadores_diarios` (retorno + ranking, por ticker/dia)
         + tabela `poc_b3_modernizacao.gold.indice_proxy` (índice-proxy + dispersão, por dia)

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# widget - modo de execucao (agendado quando disparado pelo Job, reprocessamento_manual por padrao)
dbutils.widgets.dropdown("modo_execucao", "reprocessamento_manual", ["agendado", "reprocessamento_manual"], "Modo de execucao")

In [0]:
# execucao principal - le silver, calcula 5 indicadores, grava gold, valida, registra observabilidade (com tratamento de erro)
MODO_EXECUCAO = dbutils.widgets.get("modo_execucao")

try:
    df_silver = spark.table("poc_b3_modernizacao.silver.cotacoes")
    print(f"Total de registros na Silver: {df_silver.count()}")
    display(df_silver)

    # indicador 1 - retorno diario por ticker
    df_retorno = df_silver.withColumn(
        "retorno_diario_pct",
        F.round(((F.col("preco_atual") - F.col("fechamento_anterior")) / F.col("fechamento_anterior")) * 100, 2)
    )

    # indicador 3 - ranking de valorizacao
    df_ranking = df_retorno.withColumn(
        "ranking_valorizacao",
        F.row_number().over(
            Window.partitionBy("data_referencia").orderBy(F.col("retorno_diario_pct").desc())
        )
    )
    display(df_ranking.select("ranking_valorizacao", "ticker", "data_referencia", "retorno_diario_pct").orderBy("data_referencia", "ranking_valorizacao"))

    # indicador 2 e 4 - indice-proxy e dispersao, por dia
    df_indice = df_retorno.groupBy("data_referencia").agg(
        F.round(F.avg("retorno_diario_pct"), 2).alias("indice_proxy_pct"),
        F.round(F.stddev("retorno_diario_pct"), 2).alias("dispersao_pct"),
        F.count("ticker").alias("qtd_tickers")
    )
    display(df_indice)

    merge_ou_cria(
        df_ranking.select("ticker", "data_referencia", "preco_atual", "fechamento_anterior", "retorno_diario_pct", "ranking_valorizacao"),
        "poc_b3_modernizacao.gold.indicadores_diarios",
        ["ticker", "data_referencia"]
    )
    merge_ou_cria(
        df_indice,
        "poc_b3_modernizacao.gold.indice_proxy",
        ["data_referencia"]
    )

    print("=== gold.indicadores_diarios ===")
    display(spark.table("poc_b3_modernizacao.gold.indicadores_diarios").orderBy("data_referencia", "ranking_valorizacao"))
    print("=== gold.indice_proxy ===")
    display(spark.table("poc_b3_modernizacao.gold.indice_proxy").orderBy("data_referencia"))

    # indicador 5 - indice acumulado (base 100), capitalizacao composta
    df_historico_indice = (spark.table("poc_b3_modernizacao.gold.indice_proxy")
        .select("data_referencia", "indice_proxy_pct")
        .orderBy("data_referencia")
    )

    janela_acumulada = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, 0)

    df_indice_acumulado = (df_historico_indice
        .withColumn("log_fator_diario", F.log(1 + F.col("indice_proxy_pct") / 100))
        .withColumn("log_acumulado", F.sum("log_fator_diario").over(janela_acumulada))
        .withColumn("indice_nivel", F.round(100 * F.exp(F.col("log_acumulado")), 3))
        .select("data_referencia", "indice_proxy_pct", "indice_nivel")
    )
    display(df_indice_acumulado)

    merge_ou_cria(df_indice_acumulado, "poc_b3_modernizacao.gold.indice_acumulado", ["data_referencia"])

    print("=== gold.indice_acumulado ===")
    display(spark.table("poc_b3_modernizacao.gold.indice_acumulado").orderBy("data_referencia"))

    data_referencia_gold = df_indice.agg(F.max("data_referencia")).collect()[0][0]

    registrar_execucao(
        notebook="04_gold",
        data_referencia=data_referencia_gold,
        modo_execucao=MODO_EXECUCAO,
        status="sucesso",
        inicio=inicio_execucao,
        fim=datetime.now(),
    )

except Exception as e:
    registrar_execucao(
        notebook="04_gold",
        data_referencia=None,
        modo_execucao=MODO_EXECUCAO,
        status="falha",
        inicio=inicio_execucao,
        fim=datetime.now(),
        mensagem_erro=str(e),
    )
    raise